# 01 — Raw data ingestion, identifier typing and first quality checks

First notebook in the workflow. It loads the two logical raw datasets
(`jobs` and `cars`) through the reusable package API, with identifier fields
read as nullable strings, confirms that loading succeeded, removes and counts
**completely blank rows**, validates the identifier types on the cleaned
frames, derives the **authority-backed job linkage keys**, assesses each dataset's **unique key**, checks **expected location
coverage**, reconciles each job's
**declared detail count** with the detail rows present, and validates the
jobs-to-cars **one-to-many** relationship before any join is trusted. It
performs no analysis and no other cleaning or transformation.

Pipeline order: read each CSV keeping blank lines and typing identifiers →
validate source columns → remove and count completely blank rows → validate
identifier dtypes → derive the authority-backed job linkage and offer-position keys →
assess analytical unique-key completeness and uniqueness →
check expected location coverage on the cleaned frames →
reconcile job-level declared detail counts with observed detail rows →
validate the one-to-many relationship → trusted join only if it passes →
investigate the configured expected location stream → reconcile the
source temporal fields (time-based analysis only after this passes).

Run it **top to bottom after restarting the kernel** (Kernel ▸ Restart Kernel
and Run All Cells). Every cell depends only on the cells above it.

## Confidentiality

All raw, interim and processed QL2 data is proprietary and Git-ignored.
Do not display DataFrames, columns, row counts, file names or paths in this
notebook, and clear all outputs before committing. Cell outputs here are
limited to generic status messages.

In [ ]:
import pandas as pd

from ql2_sixt_canada_analysis import (
    ANALYSIS_DATASET_DEFINITIONS,
    ANALYSIS_JOB_DETAIL_RELATIONSHIP,
    ANALYSIS_LOCATION_STREAM_COMPARISON,
    ANALYSIS_TEMPORAL_RECONCILIATION,
    EXPECTED_LOCATION_COVERAGE,
    INVESTIGATED_LOCATION_STREAM,
    COLLECTION_SCHEDULE,
    VANCOUVER_LOCATION_POLICY,
    VEHICLE_ATTRIBUTE_STABILITY,
    VehicleStabilityPreconditionError,
    DatasetKey,
    LocationCoverageConfigurationError,
    ReconciliationPreconditionError,
    RelationshipPreconditionError,
    assess_city_integrity,
    assess_collection_schedule,
    assess_dataset_location_coverage,
    assess_job_detail_join_readiness,
    assess_job_detail_reconciliation,
    assess_job_linkage,
    assess_one_to_many_join,
    assess_raw_dataset_unique_keys,
    assess_scheduled_time_coverage,
    assess_temporal_reconciliation,
    apply_location_policy,
    assess_completeness,
    assess_expected_location_streams,
    assess_location_policy,
    assess_pricing_readiness,
    assess_vehicle_attribute_stability,
    compare_location_streams,
    investigate_location_stream,
    load_job_linkage_policy,
    load_raw_datasets,
    remove_blank_rows_from_raw_datasets,
    resolve_expected_location,
    validate_raw_dataset_identifier_dtypes,
)
from ql2_sixt_canada_analysis.paths import resolve_raw_data_dir

In [ ]:
# Raw-data directory: the centralized default (data/raw under the project
# root, independent of the working directory). Automated validation points
# this at a temporary directory of synthetic CSVs through the environment
# variable documented in ql2_sixt_canada_analysis.paths. Set RAW_DIR_OVERRIDE
# to a path only for a deliberate local experiment; never commit a value.
RAW_DIR_OVERRIDE = None

raw_dir = resolve_raw_data_dir(RAW_DIR_OVERRIDE)
print("Raw-data directory selected.")

In [ ]:
# Discover both CSVs, validate their headers against the centralized column
# contracts and load them. Any problem raises an IngestionError subclass.
# Ingestion keeps blank physical lines as rows (see ingestion.RAW_CSV_READ_DEFAULTS)
# so the quality step below can measure them; nothing is cleaned here.
# Identifier columns (defined centrally in ql2_sixt_canada_analysis.schemas)
# are read as nullable strings in the same read; no manual casting is needed.
# The loader rejects any read option that could omit source records
# (complete-source policy); complete_source is True only when it held.
raw = load_raw_datasets(raw_dir)
print("Raw datasets loaded.")
print("Complete-source ingestion rules enforced:", raw.complete_source)

In [ ]:
# Confirm loading succeeded without revealing contents.
loaded = {DatasetKey.JOBS: raw.jobs, DatasetKey.CARS: raw.cars}
assert all(isinstance(frame, pd.DataFrame) for frame in loaded.values())
assert raw.jobs is not raw.cars
print("Both logical datasets are pandas DataFrames:", ", ".join(k.value for k in loaded))

## Quality step 1 — completely blank rows

A row is *completely blank* when every field is missing, an empty string or
whitespace-only; `0`, `0.0`, `False`, `"0"`, `"False"` and any other value
are meaningful, so partially blank rows are kept. The reusable function
`remove_blank_rows_from_raw_datasets` (in `ql2_sixt_canada_analysis.quality`)
applies this to `jobs` and `cars` independently, never mutates the loaded
frames, preserves column order, row order, cell values and the original index,
and returns the cleaned frames together with the original / retained / removed
counts. The counts stay in memory in `blank_rows`; they are proprietary
metrics and are deliberately not printed or written anywhere.

In [ ]:
# Transition from raw loaded data to cleaned data. ``raw`` keeps the frames
# exactly as read; ``blank_rows`` holds the per-dataset results and counts.
blank_rows = remove_blank_rows_from_raw_datasets(raw)

jobs_blank_rows = blank_rows.jobs            # BlankRowResult for jobs
cars_blank_rows = blank_rows.cars            # BlankRowResult for cars
total_blank_rows_removed = blank_rows.total_removed_blank_row_count

# Cleaned DataFrames used by everything that follows.
jobs_df = jobs_blank_rows.cleaned
cars_df = cars_blank_rows.cleaned
cleaned = blank_rows.cleaned                 # RawDatasets of the cleaned frames
print("Completely blank rows removed from both datasets; counts kept in memory.")

In [ ]:
# Confirm the step behaved as documented, without revealing any counts.
for result, frame in ((jobs_blank_rows, raw.jobs), (cars_blank_rows, raw.cars)):
    assert result.original_row_count == len(frame)
    assert result.retained_row_count == len(result.cleaned)
    assert result.original_row_count == result.retained_row_count + result.removed_blank_row_count
    assert list(result.cleaned.columns) == list(frame.columns)
assert total_blank_rows_removed == (
    jobs_blank_rows.removed_blank_row_count + cars_blank_rows.removed_blank_row_count
)
print("Blank-row counts reconcile for:", ", ".join(k.value for k in blank_rows.by_key))

## Identifier types

Identifier fields are labels, not measurements. The loader reads each
dataset's centrally defined identifier columns as pandas' nullable string
dtype, so leading zeros and long numeric-looking identifiers are preserved
exactly and missing identifiers stay missing (`<NA>`) rather than becoming
text such as `"nan"`. Raw identifier content is never rewritten; analytical
linkage uses separate derived keys (next step). The reusable
validator below confirms the types on the cleaned frames and prints nothing
about the data.

In [ ]:
# Validate identifier dtypes on the cleaned frames (silent on success;
# raises IdentifierDtypeError naming the dataset otherwise).
validate_raw_dataset_identifier_dtypes(cleaned)
print("Identifier fields use the nullable string dtype in:", ", ".join(k.value for k in DatasetKey))

## Authority-backed job linkage

The raw job identifier is **opaque text** and stays unchanged. The historical
detail export writes it with a spreadsheet decimal-zero defect, so raw text
alone cannot link details to jobs. Under the approved collection-owner
decisions in the current pricing-authority record
(`load_job_linkage_policy`; no policy unless all four job-identifier
decisions are approved), `assess_job_linkage` derives **separate**
analysis-stage keys: the job linkage key (exact match first; the only
fallback removes a final `.0` from a detail value made entirely of ASCII
digits, keeping leading zeros, and only when it identifies a unique job) and
the integer offer position. Nothing is parsed as a number, trimmed or
case-folded; missing, unmatched, ambiguous or colliding references and invalid
offer positions leave the derived key `<NA>` and block every downstream
contract. Every later step uses `analysis_jobs_df` / `analysis_cars_df` and
the analysis-stage definitions, never the raw identifier as a join key. The
cell prints aggregate counts and blocker categories only; the derived keys
are confidential technical fields.

In [ ]:
# Derive the authority-backed linkage keys (raw source columns stay unchanged).
job_linkage_policy = load_job_linkage_policy()   # None unless the job-identifier decisions are approved
job_linkage = assess_job_linkage(jobs_df, cars_df, job_linkage_policy)
job_linkage_report = job_linkage.report
job_identifier_normalization_ready = job_linkage_report.is_valid

# Analysis-stage frames used by every analytical step below.
analysis_datasets = job_linkage.datasets(cleaned)
analysis_jobs_df = analysis_datasets.jobs
analysis_cars_df = analysis_datasets.cars

print("Linkage policy:", job_linkage_report.policy_status.value)
print("Detail rows linked - exact match:", job_linkage_report.exact_match_count,
      "| approved legacy repair:", job_linkage_report.decimal_zero_repair_count)
print("Unresolved detail references - missing:", job_linkage_report.missing_identifier_count,
      "| unmatched:", job_linkage_report.unmatched_identifier_count,
      "| ambiguous:", job_linkage_report.ambiguous_identifier_count,
      "| collisions:", job_linkage_report.collision_count)
print("Parent linkage keys - invalid:", job_linkage_report.parent_key_invalid_count,
      "| duplicated rows:", job_linkage_report.parent_key_duplicate_row_count)
print("Offer positions - valid:", job_linkage_report.valid_row_index_count,
      "| legacy repairs:", job_linkage_report.legacy_row_index_repair_count,
      "| missing:", job_linkage_report.missing_row_index_count,
      "| invalid:", job_linkage_report.invalid_row_index_count)
print("Job linkage valid:", job_identifier_normalization_ready)
print("Job linkage blocked by:", ", ".join(b.value for b in job_linkage_report.blocking_reasons) or "nothing")

## Unique keys

Each dataset's business key and row grain are defined centrally in
`ql2_sixt_canada_analysis.schemas`; analytically they are the derived keys
(`ANALYSIS_DATASET_DEFINITIONS`: the job linkage key, and for details the
job linkage key plus the integer offer position). The reusable assessment checks two
separate things on the cleaned frames: **completeness** (no row is missing a
key component) and **uniqueness** (no two complete rows share the full key
tuple). It never removes, repairs or reorders rows; the reports hold counts
only and stay in memory in `key_reports`. Assessment does not stop the
notebook when a contract fails; the pass/fail status is kept in
`all_key_contracts_valid` and is deliberately not displayed.

In [ ]:
# Assess unique keys after blank-row removal and identifier validation.
# Analytical keys on the analysis-stage frames; nothing is modified by this step.
key_reports = assess_raw_dataset_unique_keys(analysis_datasets, ANALYSIS_DATASET_DEFINITIONS)

jobs_key_report = key_reports.jobs            # UniqueKeyReport for jobs
cars_key_report = key_reports.cars            # UniqueKeyReport for cars
all_key_contracts_valid = key_reports.all_valid
print("Unique-key assessment completed for:", ", ".join(k.value for k in key_reports.by_key))

## Expected location coverage

This control compares an **independent** expected-location contract
(`EXPECTED_LOCATION_COVERAGE`, defined centrally, never derived from this
extract) with the distinct locations present in the cleaned frame that the
contract names (branch-level locations live in the detail rows). It runs
before any one-to-many join. Missing expected locations, unexpected locations
and rows without a complete location are measured separately with exact
matching; no rows are removed. Expected locations are authoritative **(city,
branch) pairs**: a branch label observed under another city does not cover
the expected pair (it is an unexpected pair), and a label observed under
several cities is a conflicting assignment that fails the contract. If the
contract is unconfigured, the check fails closed: the report is `None` and
the configuration error is kept in memory. The cell prints the configured
expected and missing pairs and aggregate counts; observed source pairs are
confidential and stay in memory (`location_pair_evidence`).

In [ ]:
# Assess expected location coverage on the cleaned frame the contract names
# (before any join).
try:
    location_coverage_report = assess_dataset_location_coverage(analysis_datasets, EXPECTED_LOCATION_COVERAGE)
    location_coverage_configuration_error = None
except LocationCoverageConfigurationError as error:
    location_coverage_report = None
    location_coverage_configuration_error = error

expected_location_coverage_passed = (
    location_coverage_report is not None and location_coverage_report.is_valid
)
if location_coverage_report is not None:
    print("Expected (market, branch) pairs:",
          "; ".join(" / ".join(k) for k in location_coverage_report.expected_pairs))
    print("Missing expected pairs:",
          "; ".join(" / ".join(k) for k in location_coverage_report.missing_pairs) or "none")
    print("Observed distinct pairs:", location_coverage_report.observed_location_count,
          "| unexpected pairs:", location_coverage_report.unexpected_location_count,
          "| labels under several markets:", location_coverage_report.conflicting_location_label_count)
    print("Observed pair values: withheld (confidential)")
    print("Coverage contract passed:", expected_location_coverage_passed)  # reasons: completeness step
print("Expected-coverage step completed.")

## Job-to-detail count reconciliation

This control compares, **for each job** (jobs with zero detail rows
included, observed `0`), the number of detail rows associated with it with
**every** declared count of the centrally defined relationship
(`ANALYSIS_JOB_DETAIL_RELATIONSHIP`): the job's record count and its actual-car-rows
tally are reconciled independently and must also agree with each other; a
job is reconciled only when both match. It counts
matched, under-counted and over-counted jobs separately, plus detail rows
whose relationship key is missing or matches no job; equal global totals
cannot hide offsetting per-job errors. It never removes, repairs or
deduplicates rows. The aggregate report stays in memory in
`reconciliation_report`; if a structural precondition (such as a valid jobs
key) is not met, the report is `None` and the reason stays in memory. The
cell prints per-declaration results and the combined result (aggregates
only; per-job evidence is in memory via `job_detail_count_results`).

In [ ]:
# Reconcile declared vs observed detail counts on the cleaned frames.
# Assessment (not strict validation), so source violations do not stop the workflow.
try:
    reconciliation_report = assess_job_detail_reconciliation(analysis_jobs_df, analysis_cars_df, ANALYSIS_JOB_DETAIL_RELATIONSHIP)
    reconciliation_precondition_error = None
except ReconciliationPreconditionError as error:
    reconciliation_report = None
    reconciliation_precondition_error = error

job_detail_counts_reconciled = (
    reconciliation_report is not None and reconciliation_report.is_reconciled
)
if reconciliation_report is not None:
    for declared in reconciliation_report.count_fields:
        print(f"Declared {declared.display_name} reconciled:", declared.reconciled,
              "| matched jobs:", declared.matched_job_count,
              "| mismatched jobs:", declared.under_counted_job_count + declared.over_counted_job_count,
              "| invalid or missing:", declared.invalid_job_count + declared.missing_job_count)
    print("Declared counts agree:", reconciliation_report.declared_counts_agree)
    print("Jobs with zero detail rows:", reconciliation_report.jobs_without_linked_details_count)
    print("Combined reconciliation passed:", job_detail_counts_reconciled,
          "| blocked by:", ", ".join(reconciliation_report.violations) or "nothing")
print("Job-to-detail count reconciliation step completed.")

## One-to-many relationship

Jobs are the *one* side and cars the *many* side of the centrally defined
relationship (`ANALYSIS_JOB_DETAIL_RELATIONSHIP`): a job may have zero, one or many
detail rows, but every detail row must link to exactly one job. The contract
holds only after parent-key uniqueness, complete detail linkage (no
missing-link or orphan details), pandas `one_to_many` cardinality validation
and parent-left-join row conservation all pass. The aggregate report stays in
memory in `relationship_report`. A valid relationship alone does **not** make
a join analytically trusted (see the next step). Nothing is dropped, repaired,
displayed or written.

In [ ]:
# Validate the one-to-many relationship (the join is gated in the next step).
try:
    relationship_report = assess_one_to_many_join(analysis_jobs_df, analysis_cars_df, ANALYSIS_JOB_DETAIL_RELATIONSHIP)
    relationship_precondition_error = None
except RelationshipPreconditionError as error:
    relationship_report = None
    relationship_precondition_error = error

one_to_many_contract_valid = relationship_report is not None and relationship_report.is_valid
print("One-to-many relationship validation step completed.")

## City scope integrity

Every job must have an **assignable city** (non-null, non-blank, no
surrounding whitespace; values are never trimmed or aliased), and every
detail row linked to a job must carry **the same city as its parent job**.
Branch coverage is still evaluated from the detail rows; the job city only
proves which city's collection a job belongs to. `assess_city_integrity`
applies the single project rule and keeps `city_integrity_report` in memory.
A job with an unassignable city cannot be dropped from every stream (its
stream cannot be proven), and a cross-city row would be counted under the
wrong job's city - so either defect invalidates stream continuity,
reconciliation, the trusted join, completeness and pricing readiness. The
cell prints counts and blocker categories only; the bounded diagnostic
samples (`scope_unassignable_job_sample`, `scope_mismatch_sample`) hold
identifiers and stay in memory - never print or export them.

In [ ]:
# Assignable job scope and parent/detail agreement (one shared rule).
city_integrity_report = assess_city_integrity(
    analysis_jobs_df, analysis_cars_df, relationship=ANALYSIS_JOB_DETAIL_RELATIONSHIP, coverage=EXPECTED_LOCATION_COVERAGE
)
city_integrity_valid = city_integrity_report.is_valid
print("Jobs with unassignable scope:", city_integrity_report.scope_unassignable_job_count)
print("Linked detail rows disagreeing with their parent scope:",
      city_integrity_report.scope_mismatch_detail_row_count)
print("Scope integrity passed:", city_integrity_valid)
print("Scope integrity blocked by:",
      ", ".join(b.value for b in city_integrity_report.blocking_reasons) or "nothing")

## Trusted job-detail join

An executable join is not a trusted one: duplicated business keys or
declared counts that do not match the detail rows can coexist with a valid
relationship. `assess_job_detail_join_readiness` links only through the derived
job linkage key and requires a valid linkage report for the same frames
(`job_linkage_report`; missing, invalid or not applied blocks). It re-assesses, on the same
analysis-stage frames it joins, the jobs and detail business-key contracts,
declared-count reconciliation, the one-to-many relationship and city scope
integrity (a cross-city or unassignable row withholds the trusted join). A report that
cannot be produced counts as a failure. `job_detail_join_ready` is true only
when every contract explicitly passes; only then is
`trusted_jobs_with_details` a DataFrame (otherwise `None`).

`diagnostic_jobs_with_details` may hold the relationship-checked join when the
relationship passes but another contract fails. It is **UNTRUSTED** - for
investigation only, never for pricing, aggregation or conclusions. A trusted
join is one prerequisite; it does not make pricing analysis ready.

In [ ]:
# Fail-closed join gate: every contract must pass on the frames being joined.
job_detail_join = assess_job_detail_join_readiness(
    analysis_jobs_df, analysis_cars_df, ANALYSIS_JOB_DETAIL_RELATIONSHIP, job_linkage=job_linkage_report
)
job_detail_join_ready = job_detail_join.join_ready
trusted_jobs_with_details = job_detail_join.trusted_jobs_with_details        # None unless ready
diagnostic_jobs_with_details = job_detail_join.diagnostic_jobs_with_details  # UNTRUSTED, investigation only

# The gate re-assessed the same frames; disagreement with earlier steps means the frames changed.
if (job_detail_join.all_key_contracts_valid != all_key_contracts_valid
        or job_detail_join.declared_counts_reconciled != job_detail_counts_reconciled
        or job_detail_join.relationship_contract_valid != one_to_many_contract_valid
        or job_detail_join.city_integrity_valid != city_integrity_valid
        or job_detail_join.job_linkage_report != job_linkage_report):
    raise RuntimeError("Join-gate evidence disagrees with earlier steps; re-run the notebook in order.")

if trusted_jobs_with_details is not None:
    joined_frame_held = "trusted"
elif diagnostic_jobs_with_details is not None:
    joined_frame_held = "diagnostic only - UNTRUSTED, not for pricing, aggregation or conclusions"
else:
    joined_frame_held = "none"

print("Job linkage valid:", job_detail_join.job_linkage_valid)
print("Jobs business-key contract passed:", job_detail_join.jobs_key_contract_valid)
print("Detail business-key contract passed:", job_detail_join.details_key_contract_valid)
print("Declared counts reconciled:", job_detail_join.declared_counts_reconciled)
print("Relationship contract passed:", job_detail_join.relationship_contract_valid)
print("Scope integrity passed:", job_detail_join.city_integrity_valid)
print("Trusted join ready:", job_detail_join_ready)
print("Trusted join blocked by:", ", ".join(b.value for b in job_detail_join.blocking_reasons) or "nothing")
print("Joined frame held:", joined_frame_held)

## Expected location stream investigation

For the centrally configured stream (`INVESTIGATED_LOCATION_STREAM`, resolved
through the expected-location contract), this step traces the stream through
configuration, source presence, ingestion, cleaning, location matching,
continuity across its collection events, identifier types, parent keys,
relationship and reconciliation, and records the **earliest failing stage**
and a categorical status in `location_stream_report`. It never creates,
repairs or filters records, never applies unconfirmed aliases and never
infers a collection cadence. Continuity is counted over every in-scope
**job** of the stream's city, including jobs with zero detail rows; jobs
carry no branch, so a zero-detail job (a zero-offer capture or missing
details) cannot be assigned to the branch and makes continuity unassessable
rather than disappearing. The cell prints the job accounting (counts only);
nothing is written.

In [ ]:
# Investigate the configured expected stream (categorical report only).
stream_target = resolve_expected_location(INVESTIGATED_LOCATION_STREAM, EXPECTED_LOCATION_COVERAGE)
location_stream_report = investigate_location_stream(
    analysis_jobs_df,
    analysis_cars_df,
    stream_target,
    coverage=EXPECTED_LOCATION_COVERAGE,
    relationship=ANALYSIS_JOB_DETAIL_RELATIONSHIP,
    loaded=raw,
)
location_stream_healthy = location_stream_report.is_healthy
accounting = location_stream_report.event_accounting
print("Stream continuity:", location_stream_report.stream_continuity.value)
if accounting is not None:
    print("In-scope jobs:", accounting.in_scope_jobs,
          "| with the branch:", accounting.jobs_with_target_details,
          "| other branches only:", accounting.jobs_with_other_details_only)
    print("Zero-detail jobs:", accounting.zero_detail_jobs,
          "| zero-offer captures:", accounting.zero_offer_jobs,
          "| details missing:", accounting.missing_detail_jobs,
          "| branch identity unavailable:", accounting.branch_unassignable_jobs)
print("Expected-stream investigation step completed.")

## All expected location streams

The cell above traces one stream in detail; completeness and pricing need
**every** stream of the expected-location contract. `assess_expected_location_streams`
investigates each configured (city, branch) pair exactly once, in contract
order, and returns `expected_streams_report`, which proves the population
against the contract: an omitted, duplicated, unexpected or unavailable
report, or any partial, unassessable or unhealthy expected stream, blocks.
Row-level coverage does not substitute for job-level continuity. Adding a
pair to the contract automatically adds it here. The individual reports stay
in memory (`expected_streams_report.reports`); only counts and blocker
categories are printed.

In [ ]:
# Assess every configured expected stream (the authoritative population).
expected_streams_report = assess_expected_location_streams(
    analysis_jobs_df,
    analysis_cars_df,
    coverage=EXPECTED_LOCATION_COVERAGE,
    relationship=ANALYSIS_JOB_DETAIL_RELATIONSHIP,
    loaded=raw,
    schedule=COLLECTION_SCHEDULE,  # time coverage is assessed against this schedule only
)
all_expected_streams_healthy = expected_streams_report.all_expected_streams_healthy
print("Configured expected streams:", expected_streams_report.expected_stream_count)
print("Assessed streams:", expected_streams_report.assessed_stream_count)
print("Every expected stream assessed exactly once:", expected_streams_report.assessed_exactly_once)
print("All expected streams healthy:", all_expected_streams_healthy)
print("Expected streams blocked by:",
      ", ".join(b.value for b in expected_streams_report.blocking_reasons) or "nothing")

## Collection schedule and scheduled-time coverage

Pricing needs an **authoritative collection schedule** and `complete`
scheduled-time coverage for **every** configured expected stream.
`assess_collection_schedule` validates `COLLECTION_SCHEDULE` (`unavailable`
when none is configured, `invalid` when it cannot be used), and
`assess_scheduled_time_coverage` checks the all-expected-stream aggregate
against it: exactly one report per expected stream, assessed against that
same schedule, each with `complete` coverage. `not_assessed`,
`never_present`, `partial` and `unassessable` all fail; stream health and
temporal field trust are separate gates and never substitute for it. No
authoritative schedule exists in this repository, so the truthful result is
`unavailable` and pricing stays blocked - no schedule is ever inferred from
observed rows. Streams are numbered in contract order; labels are not
printed.

In [ ]:
# Authoritative schedule and scheduled-time coverage of every expected stream (typed results).
collection_schedule_assessment = assess_collection_schedule(COLLECTION_SCHEDULE)
scheduled_coverage_report = assess_scheduled_time_coverage(collection_schedule_assessment, expected_streams_report)
scheduled_coverage_complete = scheduled_coverage_report.all_streams_complete
print("Collection schedule:", collection_schedule_assessment.status.value)
print("Collection schedule available and valid:", collection_schedule_assessment.available)
for position, entry in enumerate(scheduled_coverage_report.stream_coverage, start=1):
    print(f"Expected stream {position} scheduled coverage:",
          entry.time_coverage.value if entry.time_coverage is not None else "no single report")
print("All expected streams complete against the schedule:", scheduled_coverage_complete)
print("Scheduled coverage blocked by:",
      ", ".join(b.value for b in scheduled_coverage_report.blocking_reasons) or "nothing")

## Temporal reconciliation

The source temporal fields are defined once in `TEMPORAL_RECONCILIATION`
(dataset, timestamp vs calendar date, format and time-zone policy). They are
parsed through the reusable API: missing, invalid and time-zone-unresolved
values are counted separately, instants are compared in UTC, reporting dates
would be derived only after converting to an authoritative reporting zone,
and naive timestamps are never given the machine's zone. Rules whose
semantics are not yet established (finish/scrape ordering, the source of
each reporting date) are reported as unavailable and keep
`temporal_fields_trusted` false; time-based analysis must not proceed until
it is true. Source values are never changed; nothing is displayed or written.

In [ ]:
# Reconcile the temporal fields on the cleaned frames (aggregate report only).
try:
    temporal_report = assess_temporal_reconciliation(analysis_jobs_df, analysis_cars_df, ANALYSIS_TEMPORAL_RECONCILIATION)
    temporal_precondition_error = None
except RelationshipPreconditionError as error:
    temporal_report = None
    temporal_precondition_error = error

# Gate for any later time-based analysis.
temporal_fields_trusted = temporal_report is not None and temporal_report.is_valid
print("Temporal reconciliation step completed.")

## Related location stream comparison

The two related streams named once in `LOCATION_STREAM_COMPARISON` are
compared through the reusable API: presence, authoritative identity metadata
(none is available in this source), shared collection events, and the exact
multisets of products and of price-aware offers in each paired capture,
checked against other stream pairs in the same scope. The categorical result
is kept in `location_comparison_report`. Identical behaviour is evidence of
duplicated collection, not proof of one physical location: the streams are
never merged, relabelled or deduplicated. Whether the two labels are one
location or two is decided only by the authority-backed
`VANCOUVER_LOCATION_POLICY` (see the pricing-readiness step below), never by
this evidence.

`likely_duplicate_streams` requires affirmative evidence for every
prerequisite: at least the configured minimum of independent paired captures
(capture events, not offer rows), complete temporal overlap with no unpaired
capture on either side, a `discriminative` scope baseline, identical
price-aware offers in every paired capture and no unidentifiable rows.
Partial overlap, an unavailable or non-discriminative baseline, too few
paired captures or any differing capture leave the result **inconclusive**,
never evidence of confirmed duplication.

Only **valid** offers are evidence: every product and price field of the
comparison signature must be present, non-blank and (for the numeric price)
a finite, non-negative number. Missing values are unassessable, never equal:
a paired capture with any invalid offer on either side is *ineligible* - it
counts neither as matching nor as differing evidence and not toward the
minimum. The scope baseline must meet the **same** standard: at least one
comparator pair needs the minimum of eligible, fully overlapping paired
captures with valid offers, and a differing capture; one partially
overlapping comparator capture never establishes it (`insufficient`). The
cell also prints eligible versus invalid pairs for the target and the
baseline, where invalid offers occurred (stream roles and field counts only,
never values), each threshold and whether inference is permitted. Even a
permitted likely duplicate remains evidence, subordinate to the
authority-backed Vancouver policy. Even a likely duplicate is
behavioural evidence only, not authoritative alias confirmation. The cell
prints the result, the aggregate evidence counts and the blocking reasons;
nothing is written.

In [ ]:
# Compare the configured related streams (categorical report only).
try:
    location_comparison_report = compare_location_streams(analysis_jobs_df, analysis_cars_df, ANALYSIS_LOCATION_STREAM_COMPARISON)
    location_comparison_precondition_error = None
except RelationshipPreconditionError as error:
    location_comparison_report = None
    location_comparison_precondition_error = error

# Diagnostic evidence only: identity is decided by VANCOUVER_LOCATION_POLICY.
if location_comparison_report is not None:
    print("Behavioural comparison result:", location_comparison_report.status.value)
    print("Paired captures:", location_comparison_report.paired_capture_count,
          "| minimum required:", location_comparison_report.minimum_paired_captures)
    print("Eligible paired captures (valid offers on both sides):",
          location_comparison_report.eligible_paired_capture_count,
          "| invalid:", location_comparison_report.invalid_paired_capture_count)
    print("Invalid offers found in:",
          ", ".join(r.value for r in location_comparison_report.invalid_evidence_streams) or "nothing")
    print("Invalid offer reasons:",
          ", ".join(r.value for r in location_comparison_report.invalid_offer_reasons) or "none")
    print("Temporal pairing sufficient:", location_comparison_report.temporal_pairing_sufficient)
    print("Offer validity sufficient:", location_comparison_report.offer_validity_sufficient)
    print("Target evidence threshold passed:", location_comparison_report.target_threshold_passed)
    print("Unpaired captures - first stream:", location_comparison_report.first_unpaired_capture_count,
          "| second stream:", location_comparison_report.second_unpaired_capture_count)
    print("Matching paired captures:", location_comparison_report.matching_paired_capture_count,
          "| differing:", location_comparison_report.differing_paired_capture_count)
    print("Temporal overlap:", location_comparison_report.temporal_overlap.value)
    print("Scope baseline:", location_comparison_report.scope_baseline.value)
    comparison_baseline = location_comparison_report.baseline_evidence
    print("Baseline paired captures - eligible:", comparison_baseline.eligible_paired_capture_count,
          "| invalid:", comparison_baseline.invalid_paired_capture_count)
    print("Baseline evidence threshold passed:", location_comparison_report.baseline_threshold_passed)
    print("Baseline discriminative:", location_comparison_report.baseline_discriminative)
    print("Duplicate-stream inference permitted:", location_comparison_report.duplicate_inference_permitted)
    print("Duplicate inference blocked by:",
          ", ".join(b.value for b in location_comparison_report.duplicate_inference_blockers) or "nothing")
print("A likely duplicate is behavioural evidence only, not authoritative alias confirmation.")
print("Related-stream comparison step completed.")

## Vehicle-attribute stability

Structural vehicle attributes are monitored independently of volatile price,
search-date and collection fields. `VEHICLE_ATTRIBUTE_STABILITY` defines the
vehicle entity and its scope, the stable attributes with their missing-value
policies, the volatile columns and the reconciled observation time; this step
runs after temporal reconciliation and only through the reusable API.
Conflicting values, missingness, insufficient history (fewer distinct
captures than the contract's minimum) and temporally unassessable entities
are counted separately in `vehicle_stability_report`.

The result describes the **full** in-scope product population: it passes
only when every entity was assessed with sufficient history and none
violated the contract. If some entities lack history the result is
`partially_assessable` (or `unassessable` if none has enough) even when no
violation was observed - an empty violations list is not proof of stability.
Proven violations are reported as `violations` and the missing-history
entities are still listed among the blocking reasons. `vehicle_attributes_stable`
is true only for a passed population. The cell prints aggregate counts and
categories; entity identifiers are confidential and are not displayed
(`classify_vehicle_entities` returns them in memory only). Values are never
rewritten, filled or dropped, and nothing is written.

In [ ]:
# Assess vehicle-attribute stability on the cleaned frame (aggregate report only).
try:
    vehicle_stability_report = assess_vehicle_attribute_stability(analysis_cars_df, VEHICLE_ATTRIBUTE_STABILITY)
    vehicle_stability_precondition_error = None
except VehicleStabilityPreconditionError as error:
    vehicle_stability_report = None
    vehicle_stability_precondition_error = error

# Gate for analysis that relies on structural vehicle attributes: the full population must pass.
vehicle_attributes_stable = vehicle_stability_report is not None and vehicle_stability_report.is_valid

if vehicle_stability_report is None:
    print("Vehicle-attribute stability unavailable:", vehicle_stability_precondition_error.reason)
else:
    print("Overall stability result:", vehicle_stability_report.status.value)
    print("Full product population valid:", vehicle_stability_report.is_valid)
    print("In-scope entities:", vehicle_stability_report.distinct_entities)
    print("Sufficient-history entities:", vehicle_stability_report.sufficient_history_entities)
    print("Insufficient-history entities:", vehicle_stability_report.insufficient_history_entities)
    print("Insufficient-history entity identifiers: withheld (confidential)")
    print("Violations:", ", ".join(vehicle_stability_report.violations) or "none observed",
          "| value-conflict entities:", vehicle_stability_report.entities_with_value_conflicts,
          "| presence-instability entities:", vehicle_stability_report.entities_with_presence_instability,
          "| same-capture conflict entities:", vehicle_stability_report.same_capture_conflict_entities)
    print("Stability blocked by:", ", ".join(vehicle_stability_report.blocking_reasons) or "nothing")
print("Vehicle-attribute stability step completed.")

## Data completeness

Completeness must be proven positively. `assess_completeness` combines the
complete-source ingestion flag, (city, branch) coverage, the all-expected-stream
aggregate (every configured stream, job-based continuity), the combined declared-count reconciliation
and city scope integrity (`city_integrity_report`). It is complete
only when every input exists and passes; each failure is a blocking reason.
Completeness does not establish timestamp authority, Vancouver identity,
key validity or stability, and does not make pricing ready by itself.

In [ ]:
# Fail-closed completeness across ingestion, coverage, continuity, counts and city scope integrity.
completeness = assess_completeness(
    datasets=analysis_datasets,
    coverage=location_coverage_report,
    streams=expected_streams_report,
    reconciliation=reconciliation_report,
    city_integrity=city_integrity_report,  # assignable job scope + parent/detail agreement
)
data_complete = completeness.complete
print("Overall completeness:", "complete" if data_complete else "not proven")
print("Completeness blocked by:", ", ".join(b.value for b in completeness.blocking_reasons) or "nothing")

## Vancouver location policy and pricing readiness

Behavioural comparison evidence (for example a likely-duplicate result) is
not identity authority. `VANCOUVER_LOCATION_POLICY` records the
authority-backed decision for the two Vancouver labels: `unresolved` (the
default - no authoritative decision exists), `confirmed_alias` or
`confirmed_distinct`. While unresolved, the labels may be neither merged nor
compared independently; a false alias flag is **not** evidence that they are
distinct. A confirmed alias requires the approved canonical analytical key
(`vancouver_analytical_locations`, which keeps the source labels for
lineage); a confirmed distinct decision permits independent comparison.

A confirmed alias is **not** usable merely because its state is
`confirmed_alias`: its canonical key must stay inside the governed scope -
the one city both Vancouver labels share - and must be one of the two
governed keys. A canonical key in Calgary (or any other city), naming
another configured stream, or any other arbitrary non-blank tuple is
rejected with a typed blocker (for example `canonical_key_crosses_governed_scope`);
the decision stays recorded but is not authority sufficient, no row is
rewritten and pricing stays blocked. The cell prints the governed keys and
city, the supplied canonical key, whether scope validation passed and
whether canonicalization is permitted.

A resolved decision is used only while it is **authority sufficient**:
authoritative identity evidence that contradicts it blocks it. A
`location_mapping_defect` (conflicting authoritative identity values within
one stream) contradicts both `confirmed_alias` and `confirmed_distinct`: the
decision stays recorded as resolved, but neither alias grouping nor
independent comparison is permitted and pricing is blocked
(`identity_evidence_conflicts_with_policy`) until the source mapping is
corrected or authoritatively reconciled. Behavioural similarity cannot
override it.

`pricing_readiness` combines this policy with every earlier gate (keys,
coverage, stream health, reconciliation, relationship, temporal trust,
vehicle stability), the authoritative schedule with `complete` coverage of
every expected stream (`scheduled_coverage_report`) and the trusted-join gate
(`job_detail_join`, whose blockers - including `join_construction_failed` -
are propagated). Pricing analysis is ready only when nothing blocks it;
the policy never overrides another gate. The cell reports the policy state,
whether it is resolved and authority sufficient, any identity-evidence
conflict, the derived permissions and the blocking reasons (categories only).

In [ ]:
# Authority-backed identity policy; comparison evidence is recorded, never decisive.
vancouver_analytical_locations = apply_location_policy(analysis_cars_df, VANCOUVER_LOCATION_POLICY)
vancouver_location_policy = assess_location_policy(
    VANCOUVER_LOCATION_POLICY, location_comparison_report, vancouver_analytical_locations
)
vancouver_location_policy_state = vancouver_location_policy.state
vancouver_location_policy_resolved = vancouver_location_policy.location_policy_resolved
# Resolved records that a decision is configured; sufficient means it is usable
# (not contradicted by authoritative identity evidence such as a mapping defect).
vancouver_location_policy_authority_sufficient = vancouver_location_policy.location_policy_authority_sufficient
# Governed scope: the alias may consolidate identity only inside the one city both keys share.
vancouver_policy_scope = vancouver_location_policy.scope
vancouver_policy_scope_valid = vancouver_location_policy.scope_valid
vancouver_canonicalization_permitted = vancouver_location_policy.canonicalization_permitted
vancouver_locations_are_aliases = vancouver_location_policy.locations_are_aliases
vancouver_locations_comparable_independently = vancouver_location_policy.locations_comparable_independently

# Fail closed: every gate must pass, and the policy must be resolved.
pricing_readiness = assess_pricing_readiness(
    location_policy=vancouver_location_policy,
    completeness=completeness,  # source, coverage, every expected stream and declared counts
    key_contracts_valid=bool(all_key_contracts_valid),
    one_to_many_contract_valid=bool(one_to_many_contract_valid),
    temporal_fields_trusted=bool(temporal_fields_trusted),
    vehicle_stability=vehicle_stability_report,
    scheduled_coverage=scheduled_coverage_report,  # authoritative schedule + every stream complete
    job_detail_join=job_detail_join,  # the validated trusted-join gate, never a non-None frame
    job_linkage=job_linkage_report,  # authority-backed linkage of the same frames
)
pricing_analysis_ready = pricing_readiness.ready

print("Vancouver identity policy state:", vancouver_location_policy_state.value)
print("Policy authority-backed and resolved:", vancouver_location_policy_resolved)
print("Identity evidence conflicts with policy:", vancouver_location_policy.identity_evidence_conflict)
print("Policy authority sufficient for analysis:", vancouver_location_policy_authority_sufficient)
print("Governed keys:", " | ".join(" / ".join(map(str, key)) for key in vancouver_policy_scope.governed_locations))
print("Governed scope:", " / ".join(vancouver_policy_scope.governed_scope or ("unavailable",)))
print("Supplied canonical key:",
      " / ".join(map(str, vancouver_policy_scope.canonical_location))
      if vancouver_policy_scope.canonical_location is not None else "none")
print("Canonical scope validation passed:", vancouver_policy_scope_valid)
print("Policy scope blocked by:", ", ".join(d.value for d in vancouver_location_policy.scope_defects) or "nothing")
print("Canonicalization permitted:", vancouver_canonicalization_permitted)
print("Vancouver labels confirmed aliases:", vancouver_locations_are_aliases)
print("Vancouver labels comparable independently:", vancouver_locations_comparable_independently)
print("Canonicalization required:", vancouver_location_policy.canonicalization_required,
      "| applied:", vancouver_location_policy.canonicalization_applied)
print("Authoritative schedule available:", pricing_readiness.schedule_available)
print("Scheduled coverage complete for every expected stream:", pricing_readiness.scheduled_coverage_complete)
print("Job linkage ready:", pricing_readiness.job_identifier_normalization_ready)
print("Trusted join ready:", pricing_readiness.trusted_join_ready)
print("Pricing analysis ready:", pricing_analysis_ready)
print("Pricing blocked by:", ", ".join(r.value for r in pricing_readiness.blocking_reasons) or "nothing")

## Next step

Further data-quality checks (coverage, timestamps, identifier consistency)
belong in reusable functions under `src/ql2_sixt_canada_analysis`, called
from the following notebook in the sequence and applied to the analysis-stage
frames `analysis_jobs_df` and `analysis_cars_df` produced above (raw identifiers
unchanged; derived linkage keys valid only when `job_identifier_normalization_ready`), with the
unique-key reports in `key_reports` and the reconciliation report in
`reconciliation_report` available for context. Analysis that needs job
attributes on detail rows may use **only** `trusted_jobs_with_details`, and
only when `job_detail_join_ready` is true (see
`job_detail_join.blocking_reasons` otherwise). A non-`None` frame is not proof
of validity: `diagnostic_jobs_with_details` is untrusted and must never feed
pricing, aggregation or conclusions, and a trusted join alone does not make
pricing ready. The stream investigation result is in
`location_stream_report`; the related-stream comparison is in
`location_comparison_report` (streams are not merged).
Coverage, stream health and job-detail reconciliation count as valid only
when `data_complete` is true: a covered branch label under the wrong city, a
zero-detail job outside the continuity count or a declared count that
disagrees with the detail rows all leave completeness unproven.
Before pricing, every job city must be assignable and every linked detail
row must carry its parent job's city (`city_integrity_valid`); while
`city_integrity_report.blocking_reasons` is non-empty the source city
assignments must be corrected upstream - no city is guessed, trimmed or
re-assigned here - and continuity, reconciliation, the trusted join,
completeness and pricing all stay blocked.
Pricing requires every configured expected stream (`expected_streams_report`),
not merely the investigated stream in `location_stream_report`; readiness
consumes `completeness`, so an unhealthy or missing expected stream blocks it.
Pricing also requires an authoritative collection schedule with `complete`
coverage for every expected stream (`scheduled_coverage_complete`; with no
schedule configured this is `unavailable` and blocks) and a ready trusted
join (`job_detail_join_ready`; a non-`None` frame is not proof).
Pricing analysis may start only when `pricing_analysis_ready` is true; check
`pricing_readiness.blocking_reasons` otherwise. **No Vancouver pricing
conclusion and no airport-versus-downtown comparison involving the Vancouver
labels may proceed while `vancouver_location_policy_state` is unresolved** -
behavioural similarity is not a decision. A partial temporal overlap, an
unavailable baseline or too few paired captures makes the comparison
inconclusive, not evidence of duplication. A confirmed alias must be analysed
only through `vancouver_analytical_locations.analytical_keys`.
Vehicle-attribute stability is in `vehicle_stability_report`; analysis relying
on structural vehicle attributes (product-level aggregation, vehicle matching,
comparisons assuming stable product identity) may proceed only when
`vehicle_attributes_stable` is true. "No violations" is not "stable": while
any product lacks sufficient history the population is only partially
assessable and **pricing analysis must not proceed**.
Time-based analysis may use temporal fields only when
`temporal_fields_trusted` is true.